In [ ]:
!pip install -q langgraph langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [2]:
import langgraph, langchain_core
from importlib.metadata import version
print("langgraph:", version("langgraph"))
print("langchain-core:", langchain_core.__version__)
print("langchain-google-genai:", version("langchain-google-genai"))

langgraph: 1.2.11
langchain-core: 1.6.3
langchain-google-genai: 4.4.0


In [3]:
import langgraph, langchain_core
from dotenv import load_dotenv
import os

load_dotenv()
print("langgraph OK, dotenv OK")
print("key loaded:", os.getenv("RESEARCHER_API_KEY") is not None)

langgraph OK, dotenv OK
key loaded: True


In [4]:
# Loading API Key For Colab
# from google.colab import userdata
# import os
# key_names = ["Researcher", "Writer", "Critic"]
# for name in key_names:
#   val = userdata.get(name)
#   os.environ[name] = val
#   print(f"{name}: {'OK, len=' + str(len(val)) if val else 'MISSING'}")

In [5]:
from typing import TypedDict, List
class ResearchState(TypedDict):
  topic: str            # User input topic
  research_notes: str   # Researcher generate info clearn up
  draft: str            # Writer current draft
  critique: str         # Critic latest judge opinion
  revision_count: int   # Current correctness times, for controling loop ending
  approved: bool        # Critic passing or not

print("State Def result complete")

State Def result complete


In [6]:
# Loading Gemini Model
from langchain_google_genai import ChatGoogleGenerativeAI
researcher_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["RESEARCHER_API_KEY"],
)

# Writer & Critic ideally use 3.5-flash
#  Due to the daily 20 limitation for 3.5-flash
#  , use 3.5-flash-lite instead

# writer_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     google_api_key=os.environ["Writer"],
# )

# critic_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     google_api_key=os.environ["Critic"],
# )

writer_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["WRITER_API_KEY"],
)

critic_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["CRITIC_API_KEY"],
)



In [7]:
# Assistant Function

def extract_text(response) -> str:
  """
    Dealing with Gemini return format (new version could be list of dict, or number)
  """
  content = response.content
  if isinstance(content, str):
    return content
  if isinstance(content, list):
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block,dict)
    )
  return str(content)

In [22]:
# Research Node Define
def researcher_node(state: ResearchState) -> dict:
  prompt = (
      f"You are a research assistant, that focuing on topic '{state['topic']}', "
      f"List out 5 to 8 key facts, background, or anything need to concider with clear, high volume info"
      "Don't include unnecessary details."
  )
  response = researcher_llm.invoke(prompt)
  return {"research_notes": extract_text(response)}

# Quick test
test_state = {"topic": "Current Taiwan electrical motorbike situation",
              "research_notes": "",
              "draft": "",
              "critique": "",
              "revision_count": 0,
              "approved": False}
result = researcher_node(test_state)
print(type(result["research_notes"]))
print(result["research_notes"][:200])


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


<class 'str'>
Here are 7 high-volume, key facts and considerations regarding the current electrical motorbike (e-motorbike/scooter) situation in Taiwan:

1. **Market Dominance and Infrastructure (Gogoro's Monopoly)


In [10]:
# Writer Node Define
def writer_node(state: ResearchState) -> dict:
  if state.get("critique"):
    # Modification Loop: regarding to Critic to modify previous draft
    prompt = (
        f"This is the draft that you wrote regarding to the topic '{state['topic']}'"
        f"Critiquer gave the opinion as follow: \n{state['critique']}\n\n"
        f"Please modify the draft, and output the corrected result."
        "Not only what you changed, but the complete draft"
    )
  else:
    # First Loop: draft according to the topic
    prompt = (
        f"You are a professional writer. Regarding to the topic: '{state['topic']}'"
        f"Write a clear structured, 300-500 words first draft: \n\n{state['research_notes']}"
    )
  response = writer_llm.invoke(prompt)
  return {
      "draft": extract_text(response),
      "revision_count": state["revision_count"] + (1 if state.get("critique") else 0),
  }

# Quick test follow the previous research_notes
test_state["research_notes"] = result["research_notes"]
draft_result = writer_node(test_state)
print(draft_result["draft"][:300])
print("revision_count:", draft_result["revision_count"])

**Powering Taiwan’s Streets: The Current State of Electric Motorbikes**

Taiwan is renowned for having one of the highest scooter densities in the world, making its streets a battleground for a green revolution. As the island accelerates toward its net-zero carbon goals, the electric motorbike (e-sc
revision_count: 0


In [11]:
# Critic Node Define
def critic_node(state: ResearchState) -> dict:
    prompt = (
        f"You are a strict checker, check the report draft regarding to the topic:'{state['topic']}':\n\n"
        f"{state['draft']}\n\n"
        f"Check if content is accurate, clear, or if there's any obvious missing point or logic issue.\n"
        f"Please use the following format to reply (first line must be APPROVED or REVISE, not other word in the first line)\n"
        f"APPROVED or REVISE\n"
        f"The following should start the opinion (if it is APPROVED, brifly explain why)"
    )
    response = critic_llm.invoke(prompt)
    text = extract_text(response).strip()

    lines = text.split("\n", 1)
    verdict = lines[0].strip().upper()
    feedback = lines[1].strip() if len(lines) > 1 else ""

    return {
        "approved": verdict.startswith("APPROVED"),
        "critique": feedback,
        "revision_count": state["revision_count"]
    }

# Quick test: use draft from previous quick test
test_state["draft"] = draft_result["draft"]
critique_result = critic_node(test_state)
print("approved:", critique_result["approved"])
print("critique:", critique_result["critique"][:300])

approved: True
critique: The report is well-researched, highly accurate regarding Taiwan's current EV landscape, and logically structured. It effectively covers the dominant players (Gogoro and Kymco), the crucial role of infrastructure and battery swapping, government subsidies, the impact of the gig economy, and the legit


In [12]:
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 3  # Temporary early stop

# Condition edge for critic
def route_after_critic(state: ResearchState) -> str:
    if state["approved"] or state["revision_count"] >= MAX_REVISIONS:
        return "end"
    return "revise"

graph_builder = StateGraph(ResearchState)
# add three nodes for researcher, writer, and critic
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("writer", writer_node)
graph_builder.add_node("critic", critic_node)

# add edges connect three state nodes
graph_builder.add_edge(START, "researcher")
graph_builder.add_edge("researcher", "writer")
graph_builder.add_edge("writer", "critic")

# If approved or reach MAX_REVISIONS -> END state
# Eles -> revise
graph_builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {"end": END, "revise": "writer"},
)

graph = graph_builder.compile()
print("Graph Complete Edit")

Graph Complete Edit


In [13]:
initial_state = {
    "topic": "AI replacing all researchers",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

final_state = graph.invoke(initial_state)

print("=== Result ===")
print("Approved:", final_state["approved"])
print("Fix round(s):", final_state["revision_count"])
print("\n=== Final Draft ===")
print(final_state["draft"])
print("\n=== Final Critic Opinion ===")
print(final_state["critique"])

=== Result ===
Approved: True
Fix round(s): 1

=== Final Draft ===
Here is the complete, revised draft incorporating the critiquer's feedback. 

Changes made to address the feedback include:
1. **Added a dedicated subsection on epistemological limits and paradigm shifts** ("The Limits of Conceptual Framing and Paradigm Shifts") to address the philosophical core of scientific creativity, intentionality, and whether AI can invent entirely new frameworks rather than just optimizing existing ones.
2. **Refined the critique on AI reliability** ("The Illusion of Omniscience: Verification and Ground-Truth Bottlenecks") to move away from generic LLM text "hallucinations" and instead focus on the deeper structural challenges of validity assessment, unexpected boundary conditions, and the need for human ground-truth verification in specialized scientific agents.

***

# The Automated Lab: Will AI Replace All Researchers?

The image is a familiar trope of modern science fiction: a sleek, sterile 

In [14]:
initial_state = {
    "topic": "AI Agent in Enterprise, how employee prevent to be laid off",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

for event in graph.stream(initial_state):
    for node_name, node_output in event.items():
        print(f"--- State: {node_name} ---")
        if "approved" in node_output:
            print(f"  approved={node_output['approved']}, revision_count={node_output['revision_count']}")
        elif "draft" in node_output:
            print(f"  draft (First 80 char): {node_output['draft'][:80]}...")
        elif "research_notes" in node_output:
            print(f"  research_notes (First 80 char): {node_output['research_notes'][:80]}...")
        print()

--- State: researcher ---
  research_notes (First 80 char): Here are 6 key facts, background contexts, and considerations for employees rega...

--- State: writer ---
  draft (First 80 char): **Surviving the Shift: How Employees Can Future-Proof Their Careers Against Ente...

--- State: critic ---
  approved=True, revision_count=0



In [15]:
def run_research_pipeline(topic: str) -> dict:
    initial_state = {
        "topic": topic,
        "research_notes": "",
        "draft": "",
        "critique": "",
        "revision_count": 0,
        "approved": False,
    }
    final_state = graph.invoke(initial_state)
    return {
        "topic": topic,
        "final_draft": final_state["draft"],
        "approved": final_state["approved"],
        "revision_count": final_state["revision_count"],
        "final_critique": final_state["critique"],
    }

# Testing
output = run_research_pipeline("How junior SWE can get hired after laid off in 2026")
print("approved:", output["approved"])
print("revision_count:", output["revision_count"])
print(output["final_draft"][:200])

approved: True
revision_count: 0
Navigating a tech layoff as a junior Software Engineer in 2026 requires more than tweaking a resume—it demands a total strategic pivot. With the tech landscape heavily reshaped by automation and AI, t


In [16]:
# Use gradio to build a simple UI
import gradio as gr

def gradio_handler(topic):
    if not topic.strip():
        return "Please enter the research topic", "", ""
    output = run_research_pipeline(topic)
    status = "✅ Approved" if output["approved"] else "⚠️ Meet the max revise rounds, Not Approved"
    meta = f"{status}| revision count:{output['revision_count']}"
    return meta, output["final_draft"], output["final_critique"]

demo = gr.Interface(
    fn=gradio_handler,
    inputs=gr.Textbox(label="Research Topic", placeholder="ex: Battery factory in 2026"),
    outputs=[
        gr.Textbox(label="Status"),
        gr.Markdown(label="Final Report"),
        gr.Textbox(label="Critic Final Opinion"),
    ],
    title="Multi-Agent Research Assistants",
    description="Researcher → Writer → Critic Cooperate Research report (Gemini 3.5 Flash-Lite)",
)

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [ ]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os

load_dotenv()
web_search = TavilySearch(
    max_results=3,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)


{'query': 'San Jose weather today', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in San Jose, CA', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'San Jose', 'region': 'California', 'country': 'United States of America', 'lat': 37.3394, 'lon': -121.8939, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1789327009, 'localtime': '2026-09-13 12:16'}, 'current': {'last_updated_epoch': 1789326000, 'last_updated': '2026-09-13 12:00', 'temp_c': 21.9, 'temp_f': 71.5, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 9.6, 'wind_kph': 15.5, 'wind_degree': 326, 'wind_dir': 'NNW', 'pressure_mb': 1016.0, 'pressure_in': 30.02, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 36, 'cloud': 0, 'feelslike_c': 18.1, 'feelslike_f': 64.6, 'windchill_c': 21.9, 'windchill_f': 71.5, 'heatindex_c': 23.8, 'heatindex_f': 74.8, 'dewpoint_c': 6.2, 'dewpoint_f': 43.2,

In [30]:
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

# System_prompt setup
system_prompt = SystemMessage(content=(
    "You are a research assistant. You have a web_search tool that can "
    "check real-time data. If the information you have is enough to "
    "answer the user's question, answer directly with text -- do not "
    "call the tool again. Only call the tool again if the current "
    "information is clearly insufficient, conflicting, or missing key details."
))

MAX_ITERATIONS = 3 # decided from Trail 5: real-data trails stopped at round 2;

def researcher_loop(question: str) -> list:
    """
    Runs the Researcher's tool-calling loop until the LLM decides it has enough information,
    or MAX_ITERATIONS is hit (safety net).
    Returns the full message history for downstream use (Writer)
    """

    messages = [
        system_prompt,
        HumanMessage(content=question),
    ]

    for round_num in range(1, MAX_ITERATIONS + 1):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # LLM judged it has enough info: stop
            return messages

        for tc in response.tool_calls:
            result = web_search.invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    # Hit the limit
    return messages

# First test "Open Question" (The question that will trigger tool recall)
print("=== Open-ended question ===")
researcher_loop("What is the weather today in San Jose, CA?")

=== Open-ended question ===


[SystemMessage(content="You are a research assistant. You have a web_search tool that can check real-time data. If the information you have is enough to answer the user's question, answer directly with text -- do not call the tool again. Only call the tool again if the current information is clearly insufficient, conflicting, or missing key details.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather today in San Jose, CA?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "weather San Jose CA today"}'}, '__gemini_function_call_thought_signatures__': {'call_5387104': 'El4KXAERTTIP7cVyEs5HlXHsVpa11rqfzSgKbKNHz4R/zC4Dmlketml0/hBu7+KVNV6z9RBFacYdN+1O+wB8yB12wEjK0/Mp4Qlgq5bAn3KPwkVbBM+YEeKou8dBH+EN'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--

In [31]:
result1 = researcher_loop("What is the weather today in San Jose, CA?")
print(result1[1].content)  # 應該印出這句，不是 "San Jose temperature today in F"

result2 = researcher_loop("What is 5 + 7?")
print(result2[1].content)
print(len(result2))  # 這題不需要查資料，應該只有 2 則訊息（human + AI 直接回答）

What is the weather today in San Jose, CA?
What is 5 + 7?
3


In [32]:
result2

[SystemMessage(content="You are a research assistant. You have a web_search tool that can check real-time data. If the information you have is enough to answer the user's question, answer directly with text -- do not call the tool again. Only call the tool again if the current information is clearly insufficient, conflicting, or missing key details.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 5 + 7?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': '5 + 7 = 12', 'extras': {'signature': 'El4KXAERTTIPVkTziBnoFQLyxdc+x+Vo+P5uiKEdxnFevQrd6kkwTHqWsaMurXpMTPapFqhPIwMJ4bhQRh6preXYurLUVNXmB77oxcJHZYRuJlSq3TubYLEGvZTXodNW'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a1ff-6e47-7be3-942a-f3fa703675c1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_